<a href="https://colab.research.google.com/github/luhipi/sarvey-tutorials/blob/main/notebooks/workshops/ISPRS26/SARvey_tutorial_ISPRS26_Taipei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p>
  <img src="https://www.uni-hannover.de/typo3conf/ext/luh_website/Resources/Public/Images/Logo/luh_logo.svg" width="128" />
  <img src="https://www.ipi.uni-hannover.de/fileadmin/site-templates/logos/ipi/ipi_logo.png" width="38" />
  <img src="https://static.wixstatic.com/media/4a7c8c_55e4ee8611e6478cbf05ded9115e3299~mv2.png/v1/fill/w_234,h_50,al_c,q_85,usm_0.66_1.00_0.01,enc_avif,quality_auto/ISPRS%202026_Horizontal%20Logo%20-%20Colour.png" width="210" />
</p>

#### InSAR Time Series Analysis with SARvey and InSAR Explorer

##### Tutorial @ [ISPRS 2026, July 2026, Toronto, Canada](https://www.isprs2026toronto.com)

##### **Metro Extension in Taipei**

---
<img src="https://seafile.projekt.uni-hannover.de/f/69eac9bb5f86400fa1ad/?dl=1" width="512">


**SARvey** is an open-source software developed for the analysis of Interferometric Synthetic Aperture Radar (InSAR) displacement time series, particularly tailored for engineering applications. It offers a comprehensive workflow to process and analyze Synthetic Aperture Radar (SAR) data, enabling users to monitor and assess ground deformations with high precision.

For more information, visit the following:
* **[SARvey GitHub Repository](https://github.com/luhipi/sarvey)**
* **[SARvey Documentation](https://sarvey.readthedocs.io/)**
* **[How to cite SARvey](https://sarvey.readthedocs.io/main/readme.html#how-to-cite/)**
* **[SARvey peer-reviewed paper](https://www.sciencedirect.com/science/article/pii/S1364815226001519)**

---

This tutorial is developed based on **[SARvey v1.3.0](https://github.com/luhipi/sarvey/tree/v1.3.0)** tag and covers the processing steps for a sample dataset, as a practical example to demonstrate its main capabilities.

---

*This tutorial is prepared by **Andreas Piter** from the [Institute of Photogrammetry and GeoInformation](https://www.ipi.uni-hannover.de/en/), Leibniz University Hannover.*



## Installation

In [ ]:
! pip install git+https://github.com/luhipi/sarvey.git@v1.3.0 --quiet

In [ ]:
! sarvey -h

In [ ]:
! apt-get -qq install tree

## Imports

In [ ]:
import os
from IPython.display import display, Image, JSON, Markdown
from matplotlib import pyplot as plt
import numpy as np
import h5py as h5
import json5 as json

## Download data

**Background**

<img src="https://seafile.projekt.uni-hannover.de/f/58ab45b8910043b18d42/?dl=1" width="512">
By Rivas et al. (2026)

---

In this tutorial, we study the displacement related to the metro extension in Taipei. The study area covers mainly urban environment with a lot of highly coherent scatterers.

The study area provided in this tutorial is slighty smaller than shown in the upper figure. We process only one line-of-sight direction (descending orbit), while in the paper (Rivas et al., 2026) the displacement was decomposed to E-W and V using both ascending and descending observations.

**Reference**\
Erik Rivas, Mahmud Haghshenas Haghighi and Mahdi Motagh (2026): *Modeling tunnel excavation in Taipei, Taiwan, using a Gaussian trough and single-look Sentinel-1 InSAR time series*. ISPRS Annals of the Photogrammetry, Remote Sensing and Spatial Information Sciences.

---

Mean amplitude image over the study area in radar coordinates and interferograms with increasing temporal baselines showing the area with displacement.

<img src="https://seafile.projekt.uni-hannover.de/f/1d906f79b929487fa5e5/?dl=1" width="512">

---

Line-of-sight cumulative displacement map from Sentinel-1 descending orbit processed with SARvey.

<img src="https://seafile.projekt.uni-hannover.de/f/74b83da48351449da866/?dl=1" width="512">

**Demo Dataset**: Metro Extension

**Dataset Highlights:**
- **Location:** Taipei, Taiwan
- **Sensor:** Sentinel-1
- **Number of images:** 392
- **Temporal interval:** 05.11.2014-28.08.2025
- **Data Type:** Coregistered stack of SLCs with corresponding geometry information

---

In [ ]:
# specify working directory. On Google Colab it should be '/content'
work_dir = '/content'

In [ ]:
# Change the directory
os.chdir(work_dir)

# Download data
! wget  https://seafile.projekt.uni-hannover.de/f/1aa0dfa74d6f413aa52b/?dl=1  -O Taipeh_S1_dsc_2014_2025.zip --show-progress -q

# Unzip data into taipeh_s1 directory
! unzip -q -o Taipeh_S1_dsc_2014_2025.zip

# Rename the extracted folder
! mv Taipeh_S1_dsc_2014_2025 taipeh_s1

# Remove the zip file
! rm Taipeh_S1_dsc_2014_2025.zip

# Define the project directory path as a variable
project_dir=os.path.join(work_dir, 'taipeh_s1')

In [ ]:
# Navigate to the project directory
os.chdir(project_dir)

# Display the directory structure in a tree-like format
! tree

# SARvey processing - Star IFG Network

Let's explore the usage of a star interferogram network which is typically used in a classical PSI processing. The default network type in SARvey, however, is "sb" (i.e. small baselines).

<img src="https://raw.githubusercontent.com/luhipi/sarvey-tutorials/main/notebooks/SARvey_tutorial_01/pics/networks_star_sb.svg" width="512">


Let's create a config file with default values!

In [ ]:
os.chdir(project_dir)

# Todo: Create a configuration file called config.json
#       If you need help, try: sarvey -h
!

We want to change a few parameters for our processing. For this, use the graphical interface of colab and open the directory on the left side. Then open the config.json using a double click in the taipeh_s1 directory and change the following parameters:

1.   Use 10 cores for the parallel processing: `general - num_cores = 10`
2.   Change the name of the output directory: `general - output_path = "outputs_star_net"`
3.   Change the interferogram network to "star": `preparation - ifg_network_type = "star"`
4.   Change the grid size for the consistency check: `consistency_check - grid_size = 100`

Safe the changes and close the config.json file.

## Step 0: Preparation

Now that we have changed the parameters, we can start our SARvey processing.
We prepare the stack of interferograms and identify coherent pixels.

In [ ]:
os.chdir(project_dir)

# Todo: Process step 0 with SARvey.
!

In [ ]:
Image(filename='outputs_star_net/pic/step_0_interferogram_network.png')

In [ ]:
# Display the output images
Image(filename='outputs_star_net/pic/step_0_amplitude_image.png')

In [ ]:
Image(filename='outputs_star_net/pic/step_0_temporal_phase_coherence.png')

❓ After the processing, check the results to answer these questions:

1.   How does the mean amplitude image look like?
2.   Are there any particularities in the interferogram network?
3.   What is the general coherence level in the study area? Are the pixels coherent in your area of interest?

Discuss your findings with your neighbours and the tutorial instructors.

Quickly note your main findings here:

1.  ...
2.  ...
3.  ...

## Step 1: Consistency

In the next step, we will create the first-order network and apply consistency checks to identify reliable pixels.

If you want to know more details about this step, please check out the Notebook on the Masjed Soleyman Dam.

In [ ]:
os.chdir(project_dir)

# Todo: Process step 1 with SARvey.
!

## Step 2: Unwrapping


Let's unwrap the interferometric phases of the first-order points and derive their displacement time series.

In [ ]:
os.chdir(project_dir)

# Todo: Process step 2 with SARvey.
!

❓Check out the displacement map of the first-order points. What is the range of the velocity estimates?

In [ ]:
# Display Estimated Velocity
Image(filename='outputs_star_net/pic/step_2_estimation_velocity.png')

## Step 3 and 4: Filtering and Densification

We now want to adjust the processing parameters for the last two steps. Apply the following change manually in the config.json:

Disable atmospheric filtering because the study area is quite small. We here assume that the effect of the atmosphere is negligable.

`filtering - apply_aps_filtering = false`

In [ ]:
os.chdir(project_dir)

# Todo: Process step 3 and 4 with SARvey.
!

❓Check out the displacement map. Does it look as expected? What is the range of the velocity values?


In [ ]:
# Display Estimated Velocities
Image(filename='outputs_star_net/pic/step_4_estimation_velocity_p2_coh80.png')

❓ Also check the estimated DEM correction of the pixels. Which pixels correspond to high-rise buildings?

In [ ]:
# Display Estimated DEM correction
Image(filename='outputs_star_net/pic/step_4_estimation_dem_correction_p2_coh80.png')

## Export to InSAR Explorer

Export and explore the displacement time series of the points in QGIS using the InSAR Explorer.

In [ ]:
os.chdir(project_dir)

! sarvey_export outputs_star_net/p2_coh80_ts.h5 -g -o outputs_star_net/taipei_metro_star_coh80.gpkg

❓Do the displacement time series match the expected displacement pattern?

<img src="https://seafile.projekt.uni-hannover.de/f/fd85918f4296448bad74/?dl=1" width="512">

❓Search for displacement time series which show a seasonal expansion of buildings.

<img src="https://seafile.projekt.uni-hannover.de/f/4998b2ab72af4aab9828/?dl=1" width="512">


❓Can you find such displacement time series? What do these jumps in the displacement time series mean?

## Interpretation of the results

❓ In the SARvey processing that we applied, we assumed a linear displacement model. Why do the displacement results not match the expected pattern? Spatially and temporally? Discuss your findings with your neigbhours and with the tutorial instructors.

# SARvey processing - Small Baseline IFG Network

The displacement signal due to the tunnel excavation is expected to be non-linear. However, we assumed a linear displacement model using the star interferogram network.

The small baseline network allows to better capture non-linear displacement patterns as we don't have to assume a particular displacement model. Instead, we assume that the displacement signal is smooth in space.

Therefore, we will now change the interferogram network to a small baseline network and re-run the processing.


<img src="https://raw.githubusercontent.com/luhipi/sarvey-tutorials/main/notebooks/SARvey_tutorial_01/pics/networks_star_sb.svg" width="512">

Change the following parameters manually for your processing in the config.json:

1.   Change interferogram network to short temporal baselines: `preparation - ifg_network_type = "stb"`
2.   Name of output directory: `general - output_path = "outputs_stb_net"`
3.   Disable temporal unwrapping: `general - apply_temporal_unwrapping = false`

The remaining parameters can stay the same.

We will now speed-up the process and run all SARvey steps one after another.

If you want to check intermediate results, you can copy the code from above.

⚠ If you copy code from above, please remember to change the path to the output directory, i.e. outputs_**stb**_net/ ! ⚠


In [ ]:
os.chdir(project_dir)

# Todo: Process steps 0 to 4 with SARvey.
!

Export and explore the displacement time series of the points in QGIS using the InSAR Explorer. Compare the results produced using a star network with the results from the small baseline network.

In [ ]:
os.chdir(project_dir)

! sarvey_export outputs_stb_net/p2_coh80_ts.h5 -g -o outputs_stb_net/taipei_metro_stb_coh80.gpkg

<img src="https://seafile.projekt.uni-hannover.de/f/8c36acec108e493eaf59/?dl=1" width="512">

If we run out of time during the tutorial, you can [here](https://seafile.projekt.uni-hannover.de/f/f1c3ea02b5fb4c23a6b6/?dl=1) download the displacement results computed using the small baseline interferogram network and explore them in QGIS using the InSAR Explorer.